# 07 - Routing RAG Index Creation

Builds the routing-only RAG vector index (Chroma) from the frozen **training split only**.
Proves `test.csv` document IDs are absent from the index. Embeddings (Gemini
`gemini-embedding-001`) are kept independent of the generation LLM (`gemini-3.6-flash`), so
another generation provider could reuse the same index later.

All logic lives in `src/newstart_ai/rag/`; this notebook only calls it, builds, and verifies.

### Load the frozen split and RAG-building tools

**Purpose:** Load all three splits and the two functions this notebook needs: one to build
the routing index, one to prove it doesn't contain test documents.

**Why this step is necessary:** The RAG index is a form of "training" the LLM+RAG method
implicitly relies on -- it must be built from `train_df` only, exactly like BERT is only
fine-tuned on `train_df`. Loading `test_df` here is intentional and safe (this notebook needs
it to *check* the index doesn't contain it, in the next few cells), but it is never passed
into the index-building function itself.

**Inputs:** `data/splits/*.csv`, `configs/rag.yaml` (embedding model, vector store choice).

**Output:** `train_df`/`val_df`/`test_df`/`manifest` and `settings`.

**How to interpret the result:** The printed line is a reminder of the rule this notebook
exists to enforce: `train_df` gets indexed, `test_df` must never be indexed.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

from newstart_ai.config import load_settings
from newstart_ai.data import load_split
from newstart_ai.rag import build_routing_index, assert_no_test_ids_in_index

settings = load_settings()
train_df, val_df, test_df, manifest = load_split(settings)
print(f"train: {len(train_df)} (indexed)   test: {len(test_df)} (must never be indexed)")
print(f"embedding model: {settings.rag.embedding_model}   vector store: {settings.rag.vector_store}")

train: 482 (indexed)   test: 151 (must never be indexed)
embedding model: gemini-embedding-001   vector store: chromadb


## Build the index from train.csv only

### Build the routing index from training documents only

**Purpose:** Convert every training-set document's text into an embedding vector (via the
Gemini embedding model) and store those vectors, along with each document's label and a text
snippet, in a persistent Chroma vector store.

**Why this step is necessary:** This index is what the LLM+RAG classifier (notebook 08)
retrieves similar examples from at prediction time. Building it from `train_df` only -- the
same rule BERT follows -- is what keeps the later LLM+RAG test-set evaluation honest: if the
index contained test documents, the retriever could return a test document's own label as a
"similar example," which would be a direct and serious form of test-set leakage.

**Inputs:** `train_df` (482 documents) and `settings.rag` (embedding model, vector store
location).

**Output:** A persisted Chroma collection on disk at `settings.rag.persist_dir`; the number
of indexed documents is printed.

**How to interpret the result:** The printed count should equal the training-set size
(482) -- confirming every training document, and nothing else, was embedded and stored.

In [2]:
num_indexed = build_routing_index(train_df, settings)
print(f"Indexed {num_indexed} training documents into '{settings.rag.persist_dir}'.")

Indexed 482 training documents into 'artifacts/vector_stores/routing_index'.


## Prove no test-set leakage

Concrete leakage check, not just a design intention: confirms none of `test.csv`'s document IDs are present in the index.

### Prove the index contains no test documents

**Purpose:** Explicitly check that none of the test set's document IDs exist inside the
just-built routing index.

**Why this step is necessary:** This is the same kind of concrete, checkable leakage proof
as `manifest.assert_no_overlap()` in notebook 03 -- rather than trusting that
`build_routing_index()` was called correctly, this cell directly inspects the index's
contents and fails loudly if a test ID is found there.

**Inputs:** `test_df`'s document IDs and the index built in the previous cell.

**Output:** Either an exception (if leakage were found) or a printed confirmation.

**How to interpret the result:** Seeing the confirmation message is what allows notebook 08
to trust the LLM+RAG test-set evaluation as a fair comparison against BERT and the plain
LLM.

In [3]:
test_ids = test_df[settings.base.dataset.id_column].astype(str).tolist()
assert_no_test_ids_in_index(settings, test_ids)
print("Confirmed: no test document IDs are present in the routing RAG index.")

Confirmed: no test document IDs are present in the routing RAG index.


## Sanity-check retrieval

Retrieves nearest neighbors for a few validation documents and shows whether the top match's label agrees with the query's own label -- a quick, informal signal that the index is semantically meaningful before it's used for evaluation in 08_llm_rag_evaluation.ipynb.

### Sanity-check retrieval quality before relying on it

**Purpose:** Pick five random validation documents, retrieve each one's top-3 nearest
neighbors from the index, and informally check whether those neighbors share the query
document's true label.

**Why this step is necessary:** Before notebook 08 uses this index for real evaluation, it's
worth confirming the embeddings are actually capturing something meaningful about agency
identity -- if retrieval mostly returned unrelated documents, that would be an early warning
that the RAG approach isn't set up correctly, before spending time and API cost on a full
evaluation. Using `val_df` (not `test_df`) for this spot-check keeps the test set fully
untouched even for this kind of informal sanity check.

**Inputs:** Five randomly sampled rows from `val_df`, and the `Retriever` built on top of
the index.

**Output:** Printed text -- for each sampled document, its true label, the labels of its
top-3 retrieved neighbors, and the top match's similarity score.

**How to interpret the result:** Seeing the retrieved labels mostly (or entirely) match the
query's own true label is a good sign that the embedding space meaningfully separates the
four agencies -- exactly what was found here (every sample's top-3 all matched). This is an
informal check, not a scored metric; the real, scored evaluation happens in notebook 08.

In [4]:
from newstart_ai.rag import Retriever

# Retriever wraps: embed the query text -> search the Chroma index -> return the top_k
# nearest neighbors with their labels and similarity scores.
retriever = Retriever(settings)
ds_cfg = settings.base.dataset

# Sampling from val_df (not test_df) keeps this informal check from ever touching the
# frozen test set, even though it's "only" a sanity check and not a scored evaluation.
sample = val_df.sample(n=5, random_state=settings.base.split.random_seed)
for _, row in sample.iterrows():
    retrieved = retriever.retrieve(row[ds_cfg.text_column], top_k=3)
    top_labels = [r.label for r in retrieved]
    print(
        f"query label={row[ds_cfg.label_column]:6s}  "
        f"top-3 retrieved labels={top_labels}  "
        f"top similarity={retrieved[0].similarity:.3f}"
    )

query label=USCIS   top-3 retrieved labels=['USCIS', 'USCIS', 'USCIS']  top similarity=0.916


query label=SSA     top-3 retrieved labels=['SSA', 'SSA', 'SSA']  top similarity=0.966


query label=SSA     top-3 retrieved labels=['SSA', 'SSA', 'SSA']  top similarity=0.822


query label=SSA     top-3 retrieved labels=['SSA', 'SSA', 'SSA']  top similarity=0.906


query label=DMV     top-3 retrieved labels=['DMV', 'DMV', 'DMV']  top similarity=0.853


## Summary for the next notebook

- Routing RAG index built from `train.csv` only ({num_indexed} documents); confirmed no
  test-set leakage.
- `08_llm_rag_evaluation.ipynb` uses this index (read-only) to evaluate the LLM+RAG
  classifier on the validation split (routing-method selection only) and the frozen test
  split (the research comparison).